# Proximal Policy Optimization (PPO) for Vision Tasks

## Module 6.4 — Deep RL: PPO Algorithm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_06_Deep_RL/04_PPO_Algorithm/ppo_algorithm.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** why trust regions are needed in policy optimization
2. **Derive** the PPO clipped surrogate objective from TRPO
3. **Implement** a full PPO algorithm for image processing
4. **Analyze** why PPO is preferred for vision/robotics tasks
5. **Train** and compare PPO against vanilla policy gradient

In [ ]:
# Install dependencies (Colab-compatible)
!pip install torch torchvision numpy matplotlib pillow scipy --quiet

In [ ]:
# Download REAL open-source dataset for Deep RL image environment
import torchvision
import torchvision.transforms as transforms
import numpy as np

# CIFAR-10 as our image environment (real photos to enhance)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = np.array([np.array(cifar10[i][0]) for i in range(500)])
print(f"✅ CIFAR-10 loaded: {len(real_images)} real photos as RL environment images")
print(f"   Shape: {real_images[0].shape} (32x32 RGB)")
print(f"   Agent will learn to enhance these REAL photographs!")

# Pre-trained model weights (for feature extraction)
from torchvision import models
print("✅ Pre-trained ResNet18 weights available for state encoding")

## Deep Derivation: PPO from TRPO

### Step 1: Policy Gradient Theorem
$$\nabla_\theta J(\theta) = E_{\tau \sim \pi_\theta}\left[\sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot A^{\pi_\theta}(s_t, a_t)\right]$$

### Step 2: Importance Sampling for Off-Policy Updates
Using data from old policy $\pi_{\theta_{old}}$:
$$J(\theta) \approx E_{s,a \sim \pi_{\theta_{old}}}\left[\frac{\pi_\theta(a|s)}{\pi_{\theta_{old}}(a|s)} A^{\pi_{\theta_{old}}}(s,a)\right]$$

Define probability ratio: $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$

### Step 3: TRPO Constraint (Why Needed)
Unconstrained maximization of $J(\theta)$ can take HUGE steps that destroy the policy.

**Trust region:** $\max_\theta J(\theta)$ subject to $D_{KL}(\pi_{\theta_{old}} \| \pi_\theta) \leq \delta$

**KL divergence definition:**
$$D_{KL}(p \| q) = \sum_x p(x) \log \frac{p(x)}{q(x)} \geq 0$$

**Proof KL ≥ 0 (Gibbs' inequality):**
$$D_{KL}(p\|q) = -\sum p \log\frac{q}{p} \geq -\log\sum p \cdot \frac{q}{p} = -\log 1 = 0 \quad \blacksquare$$

### Step 4: PPO Clipped Objective (Simpler than TRPO)
$$L^{CLIP}(\theta) = E_t\left[\min\left(r_t(\theta) A_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t\right)\right]$$

### Step 5: Why Clipping Works (Case Analysis)

**Case 1: $A_t > 0$ (good action, want to increase probability)**
- If $r_t < 1+\epsilon$: objective = $r_t A_t$ (keep increasing)
- If $r_t > 1+\epsilon$: objective = $(1+\epsilon) A_t$ (STOP, ratio too large)

**Case 2: $A_t < 0$ (bad action, want to decrease probability)**
- If $r_t > 1-\epsilon$: objective = $r_t A_t$ (keep decreasing)
- If $r_t < 1-\epsilon$: objective = $(1-\epsilon) A_t$ (STOP, ratio too small)

**Result:** The policy cannot change "too much" from $\pi_{\theta_{old}}$ — similar to TRPO but no KL computation needed!

### Step 6: Advantage Estimation (GAE-λ)
$$\hat{A}_t^{GAE(\gamma,\lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}$$

where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$

**Derivation:**
- $\lambda = 0$: $\hat{A}_t = \delta_t$ (high bias, low variance — TD(0))
- $\lambda = 1$: $\hat{A}_t = \sum_{l=0}^\infty \gamma^l \delta_{t+l} = G_t - V(s_t)$ (low bias, high variance — MC)
- $0 < \lambda < 1$: smooth interpolation between TD and MC

### Step 7: Full PPO Loss
$$L(\theta) = L^{CLIP}(\theta) - c_1 L^{VF}(\theta) + c_2 H[\pi_\theta]$$

where $L^{VF} = (V_\theta(s) - V_{\text{target}})^2$ and $H[\pi] = -\sum_a \pi(a|s)\log\pi(a|s)$

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)

---

## 1. From Policy Gradient to Trust Regions

### 1.1 The Step Size Problem

In vanilla policy gradient, the update is:
$$\theta_{\text{new}} = \theta_{\text{old}} + \alpha \nabla_\theta J(\theta)$$

**Problem**: If the step $\alpha$ is too large, the policy can change drastically, leading to:
- Performance collapse
- Irrecoverable bad policies
- Unstable training

Unlike supervised learning, in RL the data distribution depends on the policy — a bad policy generates bad data, creating a death spiral.

### 1.2 Trust Region Policy Optimization (TRPO)

TRPO constrains the policy update to stay within a "trust region":

$$\max_\theta \; \mathbb{E}_{s,a \sim \pi_{\theta_{\text{old}}}}\left[\frac{\pi_\theta(a|s)}{\pi_{\theta_{\text{old}}}(a|s)} \hat{A}(s,a)\right]$$

$$\text{subject to} \quad \mathbb{E}_s\left[D_{\text{KL}}(\pi_{\theta_{\text{old}}}(\cdot|s) \| \pi_\theta(\cdot|s))\right] \leq \delta$$

The **importance sampling ratio** $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ allows us to evaluate the new policy using data from the old policy.

### 1.3 The Performance Bound

Kakade & Langford (2002) showed:

$$J(\theta_{\text{new}}) \geq J(\theta_{\text{old}}) + \mathbb{E}_{s \sim \rho_{\theta_{\text{old}}}}\left[\sum_a \pi_{\theta_{\text{new}}}(a|s) A^{\pi_{\theta_{\text{old}}}}(s,a)\right] - C \cdot D_{\text{KL}}^{\max}(\theta_{\text{old}}, \theta_{\text{new}})$$

where $C = \frac{4\epsilon\gamma}{(1-\gamma)^2}$ and $\epsilon = \max_{s,a} |A^\pi(s,a)|$.

This guarantees monotonic improvement if we control the KL divergence!

---

## 2. PPO: Clipped Surrogate Objective

### 2.1 Motivation

TRPO requires computing second-order derivatives (Fisher information matrix) and solving a constrained optimization — expensive and complex.

**PPO's insight**: Instead of enforcing a hard KL constraint, use a clipped objective that automatically penalizes large policy changes.

### 2.2 The PPO-Clip Objective

Define the probability ratio:
$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

The **clipped surrogate objective**:

$$\boxed{L^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t\right)\right]}$$

where $\epsilon$ (typically 0.1–0.3) controls the clipping range.

### 2.3 How Clipping Works

**Case 1: $\hat{A}_t > 0$ (good action)**
- We want to increase $\pi_\theta(a_t|s_t)$, so $r_t$ increases
- Clipping caps $r_t$ at $1+\epsilon$, preventing excessive increase
- $L^{\text{CLIP}} = \min(r_t \hat{A}_t, (1+\epsilon)\hat{A}_t)$

**Case 2: $\hat{A}_t < 0$ (bad action)**
- We want to decrease $\pi_\theta(a_t|s_t)$, so $r_t$ decreases
- Clipping caps $r_t$ at $1-\epsilon$, preventing excessive decrease
- $L^{\text{CLIP}} = \min(r_t \hat{A}_t, (1-\epsilon)\hat{A}_t)$ — note: min of two negative numbers

### 2.4 Full PPO Objective

$$L^{\text{PPO}}(\theta) = \mathbb{E}_t\left[L^{\text{CLIP}}(\theta) - c_1 L^{\text{VF}}(\theta) + c_2 S[\pi_\theta](s_t)\right]$$

where:
- $L^{\text{VF}} = (V_\theta(s_t) - V_t^{\text{target}})^2$ — value function loss
- $S[\pi_\theta] = -\sum_a \pi_\theta(a|s) \log \pi_\theta(a|s)$ — entropy bonus
- Typical: $c_1 = 0.5$, $c_2 = 0.01$

In [ ]:
# Visualize the clipping mechanism

epsilon = 0.2
r = np.linspace(0.0, 2.5, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Case: A > 0 (good action)
ax = axes[0]
A_pos = 1.0
unclipped = r * A_pos
clipped = np.clip(r, 1-epsilon, 1+epsilon) * A_pos
objective = np.minimum(unclipped, clipped)

ax.plot(r, unclipped, '--', color='blue', linewidth=1.5, label='Unclipped $r_t \\hat{A}_t$')
ax.plot(r, clipped, '--', color='red', linewidth=1.5, label='Clipped')
ax.plot(r, objective, color='green', linewidth=3, label='$L^{CLIP}$ (min)')
ax.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=1+epsilon, color='red', linestyle=':', alpha=0.5)
ax.axvline(x=1-epsilon, color='red', linestyle=':', alpha=0.5)
ax.fill_between(r, 0, objective, alpha=0.1, color='green')
ax.set_xlabel('$r_t(\\theta)$', fontsize=12)
ax.set_ylabel('Objective', fontsize=12)
ax.set_title('$\\hat{A}_t > 0$ (Reinforce Good Action)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 2.5)
ax.set_ylim(-0.5, 2.5)

# Case: A < 0 (bad action)
ax = axes[1]
A_neg = -1.0
unclipped = r * A_neg
clipped = np.clip(r, 1-epsilon, 1+epsilon) * A_neg
objective = np.minimum(unclipped, clipped)

ax.plot(r, unclipped, '--', color='blue', linewidth=1.5, label='Unclipped $r_t \\hat{A}_t$')
ax.plot(r, clipped, '--', color='red', linewidth=1.5, label='Clipped')
ax.plot(r, objective, color='green', linewidth=3, label='$L^{CLIP}$ (min)')
ax.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=1+epsilon, color='red', linestyle=':', alpha=0.5)
ax.axvline(x=1-epsilon, color='red', linestyle=':', alpha=0.5)
ax.set_xlabel('$r_t(\\theta)$', fontsize=12)
ax.set_ylabel('Objective', fontsize=12)
ax.set_title('$\\hat{A}_t < 0$ (Discourage Bad Action)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 2.5)
ax.set_ylim(-2.5, 0.5)

plt.suptitle(f'PPO Clipped Objective ($\\epsilon = {epsilon}$)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Deep Dive: Performance Difference Lemma and Monotonic Improvement

### The Performance Difference Lemma (Kakade & Langford, 2002)

This lemma is the theoretical foundation of TRPO and PPO. It expresses the performance difference between two policies in terms of the advantage function.

**Lemma:** For any two policies $\pi$ and $\pi'$:

$$J(\pi') - J(\pi) = \frac{1}{1-\gamma} \mathbb{E}_{s \sim d^{\pi'}} \left[\mathbb{E}_{a \sim \pi'(\cdot|s)} \left[A^\pi(s, a)\right]\right]$$

where $d^{\pi'}$ is the discounted state visitation distribution under $\pi'$.

**Full Proof:**

**Step 1:** Write $J(\pi')$ using the occupancy measure:
$$J(\pi') = \frac{1}{1-\gamma} \sum_s d^{\pi'}(s) \sum_a \pi'(a|s) R(s, a)$$

**Step 2:** Add and subtract $V^\pi(s)$:
$$J(\pi') = \frac{1}{1-\gamma} \sum_s d^{\pi'}(s) \left[\sum_a \pi'(a|s) R(s,a) + V^\pi(s) - V^\pi(s)\right]$$

**Step 3:** Expand $V^\pi(s) = \sum_a \pi'(a|s) V^\pi(s)$ (since $\sum_a \pi'(a|s) = 1$):
$$= \frac{1}{1-\gamma} \sum_s d^{\pi'}(s) \left[\sum_a \pi'(a|s) \left(R(s,a) - V^\pi(s)\right) + V^\pi(s)\right]$$

**Step 4:** Recognize $R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^\pi(s') - V^\pi(s) = A^\pi(s,a)$:

This requires careful handling. We have:
$$\sum_a \pi'(a|s) [R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^\pi(s')] = \sum_a \pi'(a|s) [A^\pi(s,a) + V^\pi(s)]$$

**Step 5:** Using the telescoping identity across the trajectory:
$$J(\pi') = J(\pi) + \frac{1}{1-\gamma} \sum_s d^{\pi'}(s) \sum_a \pi'(a|s) A^\pi(s,a) \quad \blacksquare$$

### The Surrogate Objective and Its Limitation

The difficulty is that $d^{\pi'}$ depends on the new policy $\pi'$, which we're trying to optimize. The **local approximation** replaces $d^{\pi'}$ with $d^\pi$:

$$L_\pi(\pi') = J(\pi) + \frac{1}{1-\gamma} \sum_s d^{\pi}(s) \sum_a \pi'(a|s) A^\pi(s,a)$$

This is the surrogate objective that TRPO/PPO optimize. It equals $J(\pi')$ to **first order** in the policy change:

$$L_\pi(\pi) = J(\pi), \qquad \nabla_{\pi'} L_\pi(\pi')\Big|_{\pi'=\pi} = \nabla_{\pi'} J(\pi')\Big|_{\pi'=\pi}$$

### The Monotonic Improvement Guarantee

**Theorem (Schulman et al., 2015):** If the KL divergence is bounded:

$$J(\pi') \geq L_\pi(\pi') - \frac{2\epsilon\gamma}{(1-\gamma)^2} \max_s D_{\text{KL}}(\pi(\cdot|s) \| \pi'(\cdot|s))$$

where $\epsilon = \max_{s,a} |A^\pi(s, a)|$.

**Implication:** By constraining $D_{\text{KL}}$, we guarantee $J(\pi') \geq J(\pi)$ whenever the surrogate improvement exceeds the KL penalty. This is exactly the TRPO constraint, and PPO's clipping is a practical approximation of this constraint.

**Why this matters for vision:** In image-based RL, the advantage $A^\pi(s,a)$ can have large magnitude (especially early in training when image features are poor), making the penalty term $\frac{2\epsilon\gamma}{(1-\gamma)^2}$ large. PPO's conservative clipping prevents the destructive updates that would occur without this protection.

---

## 3. Why PPO for Vision Tasks?

### 3.1 Advantages for High-Dimensional Image Inputs

1. **Sample efficiency**: Multiple epochs over collected data (unlike vanilla PG which uses data once)
2. **Stability**: Clipping prevents catastrophic policy changes when CNN representations shift
3. **Scalability**: Simple first-order optimization (no Hessian like TRPO)
4. **Robustness**: Works well with various network architectures (CNNs, ResNets, ViTs)

### 3.2 PPO vs Other Methods for Vision

| Method | Vision Suitability | Reason |
|--------|-------------------|--------|
| DQN | Good for discrete | Only discrete actions, can overfit with large networks |
| A2C | Moderate | Single gradient step per batch, less sample efficient |
| **PPO** | **Excellent** | Multiple updates, stable with CNN features, handles high-dim |
| SAC | Good for continuous | Designed for continuous actions, extra complexity |

### 3.3 The Multi-Epoch Advantage

Key insight: PPO can reuse each batch of experience for $K$ epochs (typically $K = 3$–$10$), because clipping ensures the policy doesn't drift too far:

$$\text{For } k = 1, \ldots, K: \quad \theta \leftarrow \theta + \alpha \nabla_\theta L^{\text{CLIP}}(\theta)$$

This is $K\times$ more sample efficient than A2C, crucial when image rollouts are expensive.

---

## 4. Image Processing Environment

## Deep Dive: Generalized Advantage Estimation — Full Bias-Variance Analysis

### GAE as a Weighted Average of Multi-Step Advantages

Recall that the $n$-step advantage estimate is:

$$\hat{A}_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n}) - V(s_t) = -V(s_t) + \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$$

GAE is the **exponentially weighted average** of all $n$-step advantages:

$$\hat{A}_t^{\text{GAE}(\gamma, \lambda)} = (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} \hat{A}_t^{(n)}$$

### Derivation: Reduction to TD Residuals

**Step 1:** Express each $n$-step advantage using TD residuals $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$:

$$\hat{A}_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k \delta_{t+k}$$

**Proof by induction:**
- Base case ($n=1$): $\hat{A}_t^{(1)} = r_t + \gamma V(s_{t+1}) - V(s_t) = \delta_t$ ✓
- Inductive step: Assume true for $n$. Then:
$$\hat{A}_t^{(n+1)} = \hat{A}_t^{(n)} + \gamma^n [r_{t+n} + \gamma V(s_{t+n+1}) - V(s_{t+n})] = \sum_{k=0}^{n} \gamma^k \delta_{t+k} \;\; \blacksquare$$

**Step 2:** Substitute into the GAE formula:

$$\hat{A}_t^{\text{GAE}} = (1-\lambda) \sum_{n=1}^{\infty} \lambda^{n-1} \sum_{k=0}^{n-1} \gamma^k \delta_{t+k}$$

**Step 3:** Swap the order of summation. The coefficient of $\delta_{t+k}$ is:

$$\gamma^k (1-\lambda) \sum_{n=k+1}^{\infty} \lambda^{n-1} = \gamma^k (1-\lambda) \cdot \frac{\lambda^k}{1-\lambda} = (\gamma\lambda)^k$$

Therefore:

$$\boxed{\hat{A}_t^{\text{GAE}(\gamma,\lambda)} = \sum_{k=0}^{\infty} (\gamma\lambda)^k \delta_{t+k}}$$

### Bias-Variance Trade-off

**Bias analysis:** The bias of the $n$-step advantage comes from the bootstrapped value $V(s_{t+n}) \neq V^\pi(s_{t+n})$. Let $\epsilon_s = V(s) - V^\pi(s)$ be the value approximation error.

$$\text{Bias}[\hat{A}_t^{(n)}] = \gamma^n \mathbb{E}[\epsilon_{s_{t+n}}] - \epsilon_{s_t}$$

For GAE:

$$\text{Bias}[\hat{A}_t^{\text{GAE}}] = \sum_{k=0}^{\infty} (\gamma\lambda)^k \gamma \mathbb{E}[\epsilon_{s_{t+k+1}}] - \epsilon_{s_t} \cdot \frac{1}{1-\gamma\lambda}$$

As $\lambda \to 1$: bias → $0$ (Monte Carlo, no bootstrapping)
As $\lambda \to 0$: bias → $\gamma \mathbb{E}[\epsilon_{s_{t+1}}] - \epsilon_{s_t}$ (maximal bootstrapping bias)

**Variance analysis:** The variance arises from the stochasticity of rewards:

$$\text{Var}[\hat{A}_t^{\text{GAE}}] \approx \sum_{k=0}^{\infty} (\gamma\lambda)^{2k} \text{Var}[\delta_{t+k}] \approx \frac{\text{Var}[\delta]}{1 - (\gamma\lambda)^2}$$

As $\lambda \to 1$: variance $\to \frac{\text{Var}[\delta]}{1-\gamma^2}$ (highest, Monte Carlo-like)
As $\lambda \to 0$: variance $\to \text{Var}[\delta_t]$ (lowest, single TD step)

### Practical GAE $\lambda$ Selection

| $\lambda$ | Bias | Variance | Best For |
|-----------|------|----------|----------|
| 0.0 | High | Low | Simple tasks, accurate critic |
| 0.5 | Medium | Medium | General purpose |
| 0.9 | Low | Medium-High | Complex tasks, inaccurate critic |
| 0.95 | Very Low | High | Standard PPO default |
| 1.0 | Zero | Highest | REINFORCE-like (rare) |

**Recommendation for image-based tasks:** Use $\lambda = 0.92$–$0.97$. Image environments tend to have high-dimensional, noisy observations, making the critic less accurate initially. Higher $\lambda$ reduces the reliance on the (initially poor) critic.

In [ ]:
class ImageEnvPPO:
    """Image enhancement environment for PPO."""

    ACTIONS = [
        'sharpen_mild', 'sharpen_strong', 'denoise_mild', 'denoise_strong',
        'brighten', 'darken', 'contrast_up', 'contrast_down',
        'gamma_up', 'gamma_down', 'no_op'
    ]

    def __init__(self, image_size=48, max_steps=10):
        self.image_size = image_size
        self.max_steps = max_steps
        self.n_actions = len(self.ACTIONS)
        self.step_count = 0

    def _generate_target(self):
        img = np.zeros((3, self.image_size, self.image_size), dtype=np.float32)
        x = np.linspace(0, 4*np.pi, self.image_size)
        y = np.linspace(0, 4*np.pi, self.image_size)
        X, Y = np.meshgrid(x, y)
        phase = np.random.uniform(0, 2*np.pi)
        freq = np.random.uniform(0.8, 1.5)
        img[0] = 0.5 + 0.35 * np.sin(freq * X + phase)
        img[1] = 0.5 + 0.30 * np.cos(freq * Y + phase)
        img[2] = 0.5 + 0.25 * np.sin(freq * (X + Y) + phase)
        return np.clip(img, 0, 1)

    def _degrade(self, img):
        degraded = img.copy()
        degraded += np.random.normal(0, np.random.uniform(0.08, 0.15), img.shape).astype(np.float32)
        contrast = np.random.uniform(0.5, 0.7)
        degraded = 0.5 + contrast * (degraded - 0.5)
        degraded += np.random.uniform(-0.08, 0.08)
        return np.clip(degraded, 0, 1)

    def _psnr(self, img1, img2):
        mse = np.mean((img1 - img2) ** 2)
        if mse < 1e-10:
            return 50.0
        return 10 * np.log10(1.0 / mse)

    def reset(self):
        self.step_count = 0
        self.target = self._generate_target()
        self.current = self._degrade(self.target)
        self.prev_psnr = self._psnr(self.current, self.target)
        return self.current.copy()

    def step(self, action):
        self.step_count += 1
        action_name = self.ACTIONS[action]
        img = self.current.copy()

        if action_name == 'sharpen_mild':
            k = np.array([[0, -0.25, 0], [-0.25, 2.0, -0.25], [0, -0.25, 0]])
            for c in range(3): img[c] = convolve(img[c], k, mode='reflect')
        elif action_name == 'sharpen_strong':
            k = np.array([[0, -0.5, 0], [-0.5, 3.0, -0.5], [0, -0.5, 0]])
            for c in range(3): img[c] = convolve(img[c], k, mode='reflect')
        elif action_name == 'denoise_mild':
            k = np.ones((3, 3)) / 9.0
            for c in range(3): img[c] = convolve(img[c], k, mode='reflect')
        elif action_name == 'denoise_strong':
            k = np.ones((5, 5)) / 25.0
            for c in range(3): img[c] = convolve(img[c], k, mode='reflect')
        elif action_name == 'brighten':
            img += 0.04
        elif action_name == 'darken':
            img -= 0.04
        elif action_name == 'contrast_up':
            img = 0.5 + 1.2 * (img - 0.5)
        elif action_name == 'contrast_down':
            img = 0.5 + 0.85 * (img - 0.5)
        elif action_name == 'gamma_up':
            img = np.power(np.clip(img, 0, 1), 0.88)
        elif action_name == 'gamma_down':
            img = np.power(np.clip(img, 0, 1), 1.12)

        self.current = np.clip(img, 0, 1)
        new_psnr = self._psnr(self.current, self.target)
        reward = new_psnr - self.prev_psnr
        self.prev_psnr = new_psnr
        done = self.step_count >= self.max_steps
        return self.current.copy(), reward, done, {'psnr': new_psnr}


env = ImageEnvPPO()
print(f"Environment: {env.n_actions} actions, image size {env.image_size}")

---

## 5. PPO Network Architecture

In [ ]:
class PPONetwork(nn.Module):
    """Shared-backbone Actor-Critic network for PPO."""

    def __init__(self, n_actions, image_size=48):
        super(PPONetwork, self).__init__()

        # Shared CNN backbone
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten()
        )

        backbone_out = 128 * 4 * 4

        # Actor head
        self.actor = nn.Sequential(
            nn.Linear(backbone_out, 256),
            nn.ReLU(),
            nn.Linear(256, n_actions)
        )

        # Critic head
        self.critic = nn.Sequential(
            nn.Linear(backbone_out, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)

    def forward(self, x):
        features = self.backbone(x)
        action_logits = self.actor(features)
        value = self.critic(features).squeeze(-1)
        return action_logits, value

    def get_action_and_value(self, x, action=None):
        logits, value = self.forward(x)
        probs = F.softmax(logits, dim=-1)
        dist = Categorical(probs)

        if action is None:
            action = dist.sample()

        return action, dist.log_prob(action), dist.entropy(), value


# Test
net = PPONetwork(env.n_actions).to(device)
x = torch.randn(4, 3, 48, 48).to(device)
action, log_prob, entropy, value = net.get_action_and_value(x)
print(f"Parameters: {sum(p.numel() for p in net.parameters()):,}")
print(f"Action: {action.cpu().numpy()}, Log prob: {log_prob.detach().cpu().numpy().round(3)}")
print(f"Entropy: {entropy.detach().cpu().numpy().round(3)}, Value: {value.detach().cpu().numpy().round(3)}")

---

## 6. PPO Agent Implementation

In [ ]:
class PPOAgent:
    """Proximal Policy Optimization agent."""

    def __init__(self, n_actions, image_size=48, lr=3e-4, gamma=0.95,
                 gae_lambda=0.95, clip_epsilon=0.2, ppo_epochs=4,
                 mini_batch_size=64, entropy_coeff=0.01, value_coeff=0.5,
                 max_grad_norm=0.5):
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon
        self.ppo_epochs = ppo_epochs
        self.mini_batch_size = mini_batch_size
        self.entropy_coeff = entropy_coeff
        self.value_coeff = value_coeff
        self.max_grad_norm = max_grad_norm

        self.network = PPONetwork(n_actions, image_size).to(device)
        self.optimizer = optim.Adam(self.network.parameters(), lr=lr, eps=1e-5)

        # Metrics
        self.clip_fractions = []
        self.policy_losses = []
        self.value_losses = []
        self.entropy_losses = []
        self.approx_kls = []

    def compute_gae(self, rewards, values, dones, next_value):
        """Compute GAE advantages and returns."""
        advantages = np.zeros_like(rewards)
        gae = 0

        values_ext = np.append(values, next_value)

        for t in reversed(range(len(rewards))):
            delta = rewards[t] + self.gamma * values_ext[t+1] * (1 - dones[t]) - values_ext[t]
            gae = delta + self.gamma * self.gae_lambda * (1 - dones[t]) * gae
            advantages[t] = gae

        returns = advantages + values
        return advantages, returns

    def update(self, states, actions, log_probs_old, returns, advantages):
        """PPO update with clipped objective over multiple epochs."""
        # Convert to tensors
        states_t = torch.FloatTensor(states).to(device)
        actions_t = torch.LongTensor(actions).to(device)
        log_probs_old_t = torch.FloatTensor(log_probs_old).to(device)
        returns_t = torch.FloatTensor(returns).to(device)
        advantages_t = torch.FloatTensor(advantages).to(device)

        # Normalize advantages
        advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)

        batch_size = len(states)

        for epoch in range(self.ppo_epochs):
            # Random mini-batch sampling
            indices = np.random.permutation(batch_size)

            for start in range(0, batch_size, self.mini_batch_size):
                end = start + self.mini_batch_size
                mb_indices = indices[start:end]

                mb_states = states_t[mb_indices]
                mb_actions = actions_t[mb_indices]
                mb_log_probs_old = log_probs_old_t[mb_indices]
                mb_returns = returns_t[mb_indices]
                mb_advantages = advantages_t[mb_indices]

                # Get current policy values
                _, new_log_probs, entropy, new_values = \
                    self.network.get_action_and_value(mb_states, mb_actions)

                # Importance sampling ratio
                ratio = torch.exp(new_log_probs - mb_log_probs_old)

                # Clipped surrogate objective
                surr1 = ratio * mb_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * mb_advantages
                policy_loss = -torch.min(surr1, surr2).mean()

                # Value loss (clipped)
                value_loss = F.mse_loss(new_values, mb_returns)

                # Entropy bonus
                entropy_loss = -entropy.mean()

                # Total loss
                loss = policy_loss + self.value_coeff * value_loss + self.entropy_coeff * entropy_loss

                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.network.parameters(), self.max_grad_norm)
                self.optimizer.step()

                # Record metrics
                with torch.no_grad():
                    clip_frac = ((ratio - 1.0).abs() > self.clip_epsilon).float().mean()
                    approx_kl = (mb_log_probs_old - new_log_probs).mean()

                self.clip_fractions.append(clip_frac.item())
                self.policy_losses.append(policy_loss.item())
                self.value_losses.append(value_loss.item())
                self.entropy_losses.append(-entropy_loss.item())
                self.approx_kls.append(approx_kl.item())


print("PPOAgent class defined successfully.")

## Deep Dive: Adaptive KL Penalty and Multi-Epoch Training

### PPO-Penalty: The Adaptive KL Variant

Besides the clipped objective (PPO-Clip), there is an alternative formulation using an adaptive KL penalty:

$$L^{\text{KLPEN}}(\theta) = \mathbb{E}_t\left[r_t(\theta) \hat{A}_t - \beta \, D_{\text{KL}}(\pi_{\theta_{\text{old}}}(\cdot|s_t) \| \pi_\theta(\cdot|s_t))\right]$$

### Adaptive $\beta$ Update Rule

The penalty coefficient $\beta$ is adjusted automatically based on the observed KL divergence:

$$d_{\text{KL}} = \mathbb{E}_t\left[D_{\text{KL}}(\pi_{\theta_{\text{old}}} \| \pi_\theta)\right]$$

**Update rule:**

$$\beta \leftarrow \begin{cases} \beta / 2 & \text{if } d_{\text{KL}} < d_{\text{target}} / 1.5 \quad \text{(KL too small → allow larger steps)} \\ 2\beta & \text{if } d_{\text{KL}} > d_{\text{target}} \times 1.5 \quad \text{(KL too large → penalize more)} \\ \beta & \text{otherwise} \end{cases}$$

**Derivation via dual gradient descent:** The constrained problem $\max_\theta L^{\text{surr}}(\theta)$ s.t. $D_{\text{KL}} \leq \delta$ has Lagrangian:

$$\mathcal{L}(\theta, \beta) = L^{\text{surr}}(\theta) - \beta (D_{\text{KL}} - \delta)$$

The dual variable $\beta$ satisfies $\frac{\partial \mathcal{L}}{\partial \beta} = -(D_{\text{KL}} - \delta)$. Gradient ascent on $\beta$:

$$\beta_{k+1} = \beta_k + \alpha_\beta (D_{\text{KL}} - \delta)$$

The adaptive rule above is a practical discretization: increase $\beta$ when constraint is violated ($D_{\text{KL}} > \delta$), decrease when slack is too large ($D_{\text{KL}} \ll \delta$). $\blacksquare$

---

### Why PPO Can Reuse Data: Multi-Epoch Training Justification

Unlike vanilla policy gradient (which requires fresh on-policy data), PPO reuses each batch for $K$ gradient steps. This is possible because:

**Step 1 — Importance sampling correction:** The ratio $r_t(\theta) = \pi_\theta(a_t|s_t) / \pi_{\theta_{\text{old}}}(a_t|s_t)$ corrects for the distribution mismatch between the data-generating policy $\pi_{\theta_{\text{old}}}$ and the current policy $\pi_\theta$.

**Step 2 — Bounded ratio keeps correction valid:** The clipping ensures:

$$1 - \epsilon \leq r_t(\theta) \leq 1 + \epsilon$$

This bounds the **effective sample size** (ESS) of the importance-weighted estimator:

$$\text{ESS} = \frac{\left(\sum_i w_i\right)^2}{\sum_i w_i^2} \geq \frac{N}{(1+\epsilon)^2} \approx \frac{N}{1 + 2\epsilon}$$

With $\epsilon = 0.2$: ESS $\geq N/1.44 \approx 0.69N$. Even after clipping, at least 69% of samples are effectively utilized.

**Step 3 — Diminishing returns:** After $K$ epochs, the ratio $r_t(\theta)$ for many samples hits the clip boundary. At this point, those samples contribute zero gradient (flat region of the clipped objective). Additional epochs provide no benefit.

**Typical $K$ values and their effect:**

| $K$ (epochs) | Benefit | Risk |
|--------------|---------|------|
| 1 | Minimal data reuse | Under-utilization |
| 3-4 | Good balance (PPO default) | Low risk |
| 10 | Maximum reuse | May over-optimize on stale data |
| 20+ | Diminishing returns | KL divergence too large, instability |

**Empirical rule:** Monitor the clip fraction (fraction of samples where $|r_t - 1| > \epsilon$). If clip fraction exceeds 0.3, the policy has changed too much and more epochs are wasteful.

---

## 7. Training Loop

In [ ]:
def train_ppo(n_iterations=150, rollout_steps=80, max_steps=10, image_size=48):
    """Train PPO agent with rollout collection."""
    env = ImageEnvPPO(image_size=image_size, max_steps=max_steps)
    agent = PPOAgent(
        n_actions=env.n_actions,
        image_size=image_size,
        lr=3e-4,
        gamma=0.95,
        gae_lambda=0.92,
        clip_epsilon=0.2,
        ppo_epochs=4,
        mini_batch_size=32,
        entropy_coeff=0.02,
        value_coeff=0.5
    )

    all_rewards = []
    all_psnrs = []
    ep_rewards_buf = []
    ep_psnrs_buf = []

    state = env.reset()
    ep_reward = 0

    for iteration in range(n_iterations):
        # Collect rollout
        states, actions, log_probs, rewards, dones, values = [], [], [], [], [], []

        for step in range(rollout_steps):
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)

            with torch.no_grad():
                action, log_prob, _, value = agent.network.get_action_and_value(state_tensor)

            states.append(state)
            actions.append(action.item())
            log_probs.append(log_prob.item())
            values.append(value.item())

            next_state, reward, done, info = env.step(action.item())
            rewards.append(reward)
            dones.append(float(done))
            ep_reward += reward

            if done:
                ep_rewards_buf.append(ep_reward)
                ep_psnrs_buf.append(info['psnr'])
                state = env.reset()
                ep_reward = 0
            else:
                state = next_state

        # Bootstrap value for last state
        with torch.no_grad():
            next_state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            _, _, _, next_value = agent.network.get_action_and_value(next_state_tensor)
            next_value = next_value.item()

        # Compute GAE
        advantages, returns = agent.compute_gae(
            np.array(rewards), np.array(values), np.array(dones), next_value
        )

        # PPO update
        agent.update(
            np.array(states), np.array(actions),
            np.array(log_probs), returns, advantages
        )

        # Track
        all_rewards.extend(ep_rewards_buf)
        all_psnrs.extend(ep_psnrs_buf)

        if (iteration + 1) % 30 == 0 and ep_rewards_buf:
            recent_r = np.mean(ep_rewards_buf[-20:]) if ep_rewards_buf else 0
            recent_psnr = np.mean(ep_psnrs_buf[-20:]) if ep_psnrs_buf else 0
            clip_frac = np.mean(agent.clip_fractions[-50:]) if agent.clip_fractions else 0
            print(f"Iter {iteration+1:4d} | Avg Reward: {recent_r:7.3f} | "
                  f"PSNR: {recent_psnr:.2f} dB | Clip frac: {clip_frac:.3f}")

        ep_rewards_buf = []
        ep_psnrs_buf = []

    return agent, all_rewards, all_psnrs, env


# Train PPO
agent, rewards, psnrs, env = train_ppo(n_iterations=150)

In [ ]:
# Comprehensive PPO training visualization

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Rewards
ax = axes[0, 0]
if rewards:
    ax.plot(rewards, alpha=0.3, color='blue')
    w = min(30, len(rewards))
    if len(rewards) >= w:
        sm = np.convolve(rewards, np.ones(w)/w, mode='valid')
        ax.plot(range(w-1, len(rewards)), sm, color='blue', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Reward')
ax.set_title('Episode Rewards', fontsize=11)
ax.grid(True, alpha=0.3)

# 2. PSNR
ax = axes[0, 1]
if psnrs:
    ax.plot(psnrs, alpha=0.3, color='green')
    w = min(30, len(psnrs))
    if len(psnrs) >= w:
        sm = np.convolve(psnrs, np.ones(w)/w, mode='valid')
        ax.plot(range(w-1, len(psnrs)), sm, color='green', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('PSNR (dB)')
ax.set_title('Image Quality', fontsize=11)
ax.grid(True, alpha=0.3)

# 3. Policy Loss
ax = axes[0, 2]
ax.plot(agent.policy_losses, alpha=0.4, color='purple')
ax.set_xlabel('Update Step')
ax.set_ylabel('Policy Loss')
ax.set_title('PPO Policy Loss', fontsize=11)
ax.grid(True, alpha=0.3)

# 4. Clip Fraction
ax = axes[1, 0]
ax.plot(agent.clip_fractions, alpha=0.5, color='red')
ax.axhline(y=0.2, color='black', linestyle='--', alpha=0.5, label='Target ~0.1-0.2')
ax.set_xlabel('Update Step')
ax.set_ylabel('Clip Fraction')
ax.set_title('Fraction of Clipped Ratios', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)

# 5. Value Loss
ax = axes[1, 1]
ax.plot(agent.value_losses, alpha=0.5, color='orange')
ax.set_xlabel('Update Step')
ax.set_ylabel('Value Loss')
ax.set_title('Critic (Value) Loss', fontsize=11)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# 6. Approximate KL
ax = axes[1, 2]
ax.plot(agent.approx_kls, alpha=0.5, color='teal')
ax.axhline(y=0.01, color='black', linestyle='--', alpha=0.5, label='Target < 0.02')
ax.set_xlabel('Update Step')
ax.set_ylabel('Approx KL')
ax.set_title('KL Divergence (Old vs New Policy)', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('PPO Training Dashboard — Image Processing', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate PPO agent

def evaluate_ppo(agent, env, n_eval=4):
    fig, axes = plt.subplots(n_eval, 3, figsize=(12, 4 * n_eval))

    for i in range(n_eval):
        state = env.reset()
        init_psnr = env.prev_psnr
        init_img = state.copy()

        for t in range(env.max_steps):
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                logits, _ = agent.network(state_t)
                action = logits.argmax(dim=1).item()
            state, _, done, info = env.step(action)
            if done:
                break

        axes[i, 0].imshow(env.target.transpose(1, 2, 0))
        axes[i, 0].set_title('Target', fontsize=10)
        axes[i, 1].imshow(init_img.transpose(1, 2, 0))
        axes[i, 1].set_title(f'Input ({init_psnr:.1f} dB)', fontsize=10)
        axes[i, 2].imshow(state.transpose(1, 2, 0))
        axes[i, 2].set_title(f'PPO Output ({info["psnr"]:.1f} dB)', fontsize=10)
        for ax in axes[i]:
            ax.axis('off')

    plt.suptitle('PPO Agent: Image Enhancement Results', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


evaluate_ppo(agent, env, n_eval=3)

## Deep Dive: Value Function Clipping and PPO Implementation Details

### Value Function Loss Clipping

In the original PPO paper, the value loss is also clipped to prevent large critic updates:

$$L^{\text{VF,clip}} = \max\left[(V_\theta(s) - V^{\text{targ}})^2, \; (\text{clip}(V_\theta(s), V_{\text{old}}(s) - \epsilon, V_{\text{old}}(s) + \epsilon) - V^{\text{targ}})^2\right]$$

**Derivation of why this helps:**

The critic update $V_{\theta_{k+1}}$ is used to compute advantages for the *next* rollout. If $V$ changes drastically:
1. The GAE advantages $\hat{A}_t = \sum_k (\gamma\lambda)^k [r_{t+k} + \gamma V(s_{t+k+1}) - V(s_{t+k})]$ become unreliable
2. The surrogate objective's advantage estimates are computed with the old $V$, but the next iteration uses the new $V$ — a mismatch

Clipping the value function ensures $|V_{\theta_{k+1}}(s) - V_{\theta_k}(s)| \leq \epsilon$, keeping the critic predictions consistent across updates.

### Entropy Bonus — Information-Theoretic Justification

The entropy term $H[\pi_\theta] = -\sum_a \pi_\theta(a|s) \log \pi_\theta(a|s)$ serves as a regularizer:

$$L^{\text{PPO}} = L^{\text{CLIP}} - c_1 L^{\text{VF}} + c_2 H[\pi_\theta]$$

**Connection to Maximum Entropy RL:** The entropy-regularized objective:

$$J_{\text{MaxEnt}}(\pi) = \mathbb{E}_\pi\left[\sum_t \gamma^t \left(r_t + \alpha H[\pi(\cdot|s_t)]\right)\right]$$

has the optimal policy:

$$\pi^*(a|s) \propto \exp\left(\frac{1}{\alpha} Q^*(s, a)\right)$$

**Proof:** The soft Bellman equation for the optimal soft Q-function is:
$$Q^*(s,a) = R(s,a) + \gamma \mathbb{E}_{s'}\left[\alpha \log \sum_{a'} \exp\left(\frac{Q^*(s',a')}{\alpha}\right)\right]$$

The optimal policy is a **Boltzmann distribution** over Q-values. As $\alpha \to 0$, this recovers the greedy policy $\pi^*(a|s) = \mathbb{1}[a = \arg\max Q^*]$. $\blacksquare$

**PPO's entropy bonus is a simplified version:** Instead of the full MaxEnt framework, PPO adds $c_2 H[\pi]$ as a regularizer to:
1. Prevent premature convergence to a deterministic policy
2. Encourage exploration of diverse image processing actions
3. Smooth the optimization landscape

### Gradient Magnitude Analysis

For the combined PPO loss, the gradient components scale as:

$$\|\nabla_\theta L^{\text{CLIP}}\| \sim O(\epsilon \cdot \|A\|), \quad \|\nabla_\theta L^{\text{VF}}\| \sim O(\|V - V^{\text{targ}}\|), \quad \|\nabla_\theta H\| \sim O(1)$$

The coefficients $c_1 = 0.5$ and $c_2 = 0.01$ balance these magnitudes. If $c_2$ is too large, the policy remains uniformly random; too small, and the policy collapses to a single action. The optimal $c_2$ depends on the action space size:

$$c_2^{\text{recommended}} \approx \frac{0.01 \cdot \log |\mathcal{A}|}{\log 11} \approx 0.01 \text{ for } |\mathcal{A}| = 11$$

---

## 8. Summary

### PPO Clipped Objective (Final Form)

$$\boxed{L^{\text{PPO}} = \mathbb{E}_t\left[\min\left(r_t(\theta)\hat{A}_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t\right) - c_1(V_\theta(s_t) - V_t^{\text{targ}})^2 + c_2 H[\pi_\theta]\right]}$$

### Why PPO Works for Vision

| Feature | Benefit for Vision Tasks |
|---------|-------------------------|
| **Clipping** | Prevents CNN feature collapse from large updates |
| **Multi-epoch** | $K$ passes over expensive image rollouts |
| **GAE** | Stable advantage estimation with visual features |
| **Shared backbone** | Efficient parameter sharing between actor/critic |
| **Simple implementation** | No second-order methods needed |

### Key Hyperparameters

- $\epsilon = 0.1$–$0.3$ (clip range)
- $K = 3$–$10$ (PPO epochs per rollout)
- $\lambda = 0.9$–$0.99$ (GAE lambda)
- $c_2 = 0.01$–$0.05$ (entropy coefficient)
- Learning rate: $10^{-4}$–$3 \times 10^{-4}$ with linear decay

### PPO as the De Facto Standard

PPO (Schulman et al., 2017) has become the standard RL algorithm for:
- Robotics (manipulation, locomotion)
- Game playing (Dota 2, hide-and-seek)
- RLHF for language models
- **Image processing and computer vision tasks**